# Somo la 13 - Kumbukumbu ya Wakala


## Usanidi

Daftari hili linaonyesha jinsi ya kujenga wakala wa uhifadhi wa safari kwa kutumia **kumbukumbu ya kudumu** kwa kutumia **Microsoft Agent Framework** (MAF).

Utajifunza jinsi aina tofauti za kumbukumbu za wakala — za kazi, za muda mfupi, na za muda mrefu — zinavyoathiri jinsi wakala anavyohifadhi na kutumia taarifa katika mazungumzo.

**Masharti ya awali:**
- Mradi wa Microsoft Foundry ulio na mfano wa mazungumzo uliowekwa (mfano `gpt-4.1-mini`).
- Kuwa umeingia kutumia Azure CLI — tumia `az login` kwenye terminal yako.
- `AZURE_AI_PROJECT_ENDPOINT` — anwani ya mradi wako wa Microsoft Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — jina la mfano uliowekwa.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Aina za Kumbukumbu za Wakala

Maajenti wa AI wanaweza kutumia aina tofauti za kumbukumbu, kila moja ikiwa na madhumuni tofauti:

### Kumbukumbu ya Kazi
Mfululizo wa mazungumzo yenyewe — ujumbe uliobadilishanwa katika kikao kimoja. Wakala anaweza kurejea nyuma kwa ujumbe za awali katika mfululizo huo huo ili kudumisha muendelezo. Katika MAF hii huundwa kupitia **`agent.create_session()`**, ambayo inarudisha `AgentSession`.

### Kumbukumbu ya Muda Mfupi
Taarifa zinazodumu kwa muda wa kazi au kikao lakini hazihifadhiwi kwa kudumu. Kwa mfano, wakala anaweza kukusanya ukweli wakati wa mazungumzo ya kupanga yenye mizunguko mingi na kuyatumia kutoa ratiba ya mwisho.

### Kumbukumbu ya Muda Mrefu
Upendeleo na ukweli unaodumu **katika vikao vyote**. Mtumiaji anayerudi si lazima arudie vizingiti vyao vya lishe au mtindo wa safari. Kumbukumbu ya muda mrefu kawaida huungwa mkono na hifadhi ya nje — hifadhidata, faili, au faharasa la vekta — na kuletwa kwa wakala kupitia zana.


## Kumbukumbu Inayofanya Kazi na Vikao

Aina rahisi ya kumbukumbu ni kikao cha mazungumzo. Unapopita kikao kilekile (kilichoundwa kupitia `agent.create_session()`) kwa wito mfululizo wa `agent.run()`, wakala anaona historia yote ya mazungumzo hayo na anaweza kukumbuka maelezo ya awali.

Acheni tuunde wakala wa usafiri na kuonyesha kumbukumbu inayofanya kazi.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Wakala alikumbuka bajeti vizuri kwa sababu ujumbe wote huwa katika kikao kimoja. Hii ni **kumbukumbu ya kazi** — hupatikana tu kwa muda wa kikao.

### Nini hutokea na kipande kipya cha mazungumzo?

Ikiwa tunaunda kikao **kipya**, wakala hana kumbukumbu ya mazungumzo ya awali:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Mfano wa Kumbukumbu ya Muda Mrefu

Ili kukumbuka mapendeleo ya mtumiaji **katika vipindi tofauti**, tunahitaji hifadhi ya kudumu inayojitokeza nje ya mzungumzo. Wakala hufikia hifadhi hii kupitia **zana** — kazi ambazo anaweza kuitumia kuhifadhi na kuchukua taarifa.

Hapo chini tunatekeleza hifadhi rahisi ya upendeleo ndani ya kumbukumbu (katika uzalishaji ungefanya hivi kwa kutumia hifadhidata au kiashiria cha vekta) na kuitangaza kama zana zinazotumiwa na wakala.

### Mimarisho
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Tukio 1 — Mtumiaji wa mara ya kwanza anahifadhi safari ya kumbukumbu

Sarah anatembelea kwa mara ya kwanza. Wakala anapaswa kuhifadhi mapendeleo yake kupitia zana na kuyatumia kupendekeza hoteli.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Hali ya 2 — Sarah anarudi baada ya wiki

Sarah anaanza **mabadiliko mapya kabisa** (akifanya kama kikao kipya). Kumbukumbu ya kazi ni tupu, lakini duka la mapendeleo ya muda mrefu bado lina taarifa zake. Wakala anapaswa kuzipata na kuzitumia kubinafsisha mapendekezo.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Muhtasari

Katika somo hili ulijifunza aina tatu za kumbukumbu za wakala na jinsi ya kuzitekeleza kwa kutumia Microsoft Agent Framework:

| Aina ya Kumbukumbu | Mbinu ya MAF | Muda wa Kuishi |
|---|---|---|
| **Inayofanya kazi** | `agent.create_session()` | Mazungumzo moja |
| **Muda mfupi** | Muktadha uliokusanywa ndani ya mfululizo | Kazi/mojawapo wa mkutano |
| **Muda mrefu** | Hifadhidata ya nje inayopatikana kwa kutumia `@tool` shughuli | Kuendelea kwa vikao |

### Mambo Muhimu ya Kumbuka
1. **`agent.create_session()`** hutoa kumbukumbu ya kazi — wakala anaona historia kamili ya mazungumzo ndani ya kikao.
2. **Vikao vipya hupoteza muktadha** — bila kumbukumbu ya muda mrefu wakala hawezi kukumbuka mazungumzo ya zamani.
3. **Shughuli za `@tool`** zinavuka pengo — zinamwezesha wakala kuhifadhi na kupata taarifa kutoka kwa hifadhidata ya kudumu.
4. **Ubinafsishaji unaboreshwa kwa wakati** — kadri upendeleo unavyohifadhiwa, ndivyo mapendekezo ya wakala yanavyozidi kuwa bora.

### Matumizi Halisi Duniani
- **Huduma kwa Wateja**: Kukumbuka historia na upendeleo wa mteja
- **Msaidizi Binafsi**: Kuhifadhi muktadha kwa siku au wiki
- **Huduma za Afya**: Kufuatilia taarifa na upendeleo wa mgonjwa
- **Biashara Mtandaoni**: Ununuzi binafsi kulingana na historia

### Hatua Zifuatazo
- Badilisha kamusi ya kumbukumbu ya ndani na hifadhidata au duka la vektor (k.m. Azure AI Search)
- Ongeza muda wa kumalizika kwa kumbukumbu kwa taarifa nyeti kwa wakati
- Tengeneza mifumo ya wakala wengi yenye kumbukumbu za pamoja
- Chunguza daftari la Cognee kwa kumbukumbu inayotegemea mti wa maarifa


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
